In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer
from linearmodels.iv import IV2SLS, compare
from IPython.display import display, HTML

In [ ]:
def get_labels(data):
    stata_reader = pd.read_stata(data, iterator=True)
    labels_dict = stata_reader.variable_labels()
    data_dictionary = pd.DataFrame(columns=["Variable", "Label"], data=list(labels_dict.items()))
    return data_dictionary

In [ ]:
df = pd.read_stata("../Data/Acemoglu/AJR_dataset.dta")
df_labels = get_labels("../Data/Acemoglu/AJR_dataset.dta")

## Income and settler mortality

In [ ]:


sns.regplot(data=df, x="logem4", y="logpgp95",ci=None, scatter=False)
sns.despine()
plt.title(" FIGURE 1. REDUCED-FORM RELATIONSHIP BETWEEN INCOME AND SETTLER MORTALITY", fontname="Times", )
plt.xlabel("Log of Settler Mortality")
plt.ylabel("Log GDP per capita, PPP, 1995")
plt.ylim(4,11)

for mort, gdp in dict(zip(df["logem4"], df["logpgp95"])).items():
    country = df.loc[(df["logpgp95"] == gdp) & (df["logem4"] == mort), "shortnam"]
    plt.text(s=str(country)[7:10],
             x=mort, y=gdp)
    
plt.show()

## Table 2

In [ ]:
regressors = ["avexpr", "lat_abst", "asia", "africa", "other_cont"]

reg1 = smf.ols(formula = "logpgp95 ~ avexpr", data=df).fit()
reg2 = smf.ols(formula = "logpgp95 ~ avexpr", data=df.query("baseco==1")).fit()

reg3 = smf.ols(formula = "logpgp95 ~ avexpr + lat_abst", data=df).fit()
reg4 = smf.ols(formula = f"logpgp95 ~ avexpr + {'+'.join(regressors)}", data=df).fit()

reg5 = smf.ols(formula = "logpgp95 ~ avexpr + lat_abst", data=df.query("baseco==1")).fit()
reg6 = smf.ols(formula = f"logpgp95 ~ avexpr + {'+'.join(regressors)}", data=df.query("baseco==1")).fit()

reg7 = smf.ols(formula = "loghjypl ~ avexpr", data=df).fit()
reg8 = smf.ols(formula = "loghjypl ~ avexpr", data=df.query("baseco==1")).fit()

models=[reg1, reg2, reg3, reg4, reg5, reg6, reg7, reg8]

In [ ]:
table2 = Stargazer(models)

In [ ]:
cov_labels = ["Average protection against expropriation risk, 1985-1995",
              "Latitude", "Asia dummy", "Africa dummy", "Other continent dummy"]
table2.covariate_order(regressors)
table2.rename_covariates(dict(zip(regressors, cov_labels)))
table2.custom_columns(["Whole world", "Base sample", "Whole world","Whole world", "Base sample", "Base sample", "Whole world", "Base sample"])
table2.show_model_numbers(False)
table2.dep_var_name = "Log GDP per capita (1-6) /   Log output per worker in 1998 (7-8)"
table2.covariate_order(regressors)
table2.show_adj_r2 = False
table2.show_confidence_intervals(False)
table2.show_degrees_of_freedom(False)
table2.show_f_statistic = False
table2.show_residual_std_err = False
table2

## IV-2SLS Regression

In [ ]:
base_sample = df.query("baseco == 1")
bs_without_neo_europes = base_sample.query("rich4 == 0")
bs_without_africa = base_sample.query("africa == 0")
samples = [base_sample, bs_without_neo_europes, bs_without_africa]

### OLS regression

In [ ]:
regressors =  [["avexpr"], ["avexpr", "lat_abst"]]
with_continents = [["avexpr","asia", "africa","other_cont" ],
                   ["avexpr", "lat_abst","asia", "africa","other_cont"]] 
ols_models = []

for sample in samples:
    for indep in regressors:
        reg = smf.ols(formula=f"logpgp95 ~ {'+'.join(indep)}", data=sample).fit()
        ols_models.append(reg)
        
for indep in with_continents:
    reg = smf.ols(formula=f"logpgp95 ~ {'+'.join(indep)}", data=base_sample).fit()
    ols_models.append(reg)

reg_worker = smf.ols(formula="loghjypl ~ avexpr", data=base_sample).fit()
ols_models.append(reg_worker)

table4_ols = Stargazer(ols_models)
table4_ols.covariate_order(["avexpr"])
table4_ols.show_adj_r2 = False
table4_ols.show_confidence_intervals(False)
table4_ols.show_degrees_of_freedom(False)
table4_ols.show_f_statistic = False
table4_ols.show_residual_std_err = False
table4_ols.custom_columns(["Base Sample", "Base Sample", "Base sample without Neo-Europes","Base sample without Neo-Europes", "Base sample without Africa", "Base sample without Africa", "Base sample with continent dummies", "Base sample with continent dummies", "Base sample, dependet variable is log output per worker"])
table4_ols.show_r2 = False
table4_ols.show_notes = False
table4_ols.dep_var_name = "Log GDP per Capita"

table4_ols

### First Stage

In [ ]:
fs_models = []

for sample in samples:
    for indep in regressors:
        reg = smf.ols(formula=f"avexpr ~ {'+'.join(indep)}", data=sample).fit()
        fs_models.append(reg)
        
for indep in with_continents:
    reg = smf.ols(formula=f"avexpr ~ {'+'.join(indep)}", data=base_sample).fit()
    fs_models.append(reg)

reg_worker = smf.ols(formula="loghjypl ~ logem4", data=base_sample).fit()
fs_models.append(reg_worker)

table4_fs = Stargazer(fs_models)
table4_fs.covariate_order(with_continents[1])
table4_fs.show_adj_r2 = False
table4_fs.show_confidence_intervals(False)
table4_fs.show_degrees_of_freedom(False)
table4_fs.show_f_statistic = False
table4_fs.show_residual_std_err = False
table4_fs.custom_columns(["Base Sample", "Base Sample", "Base sample without Neo-Europes","Base sample without Neo-Europes", "Base sample without Africa", "Base sample without Africa", "Base sample with continent dummies", "Base sample with continent dummies", "Base sample, dependet variable is log output per worker"])
table4_fs.dep_var_name = "Log GDP per Capita"
table4_fs.title(" First Stage for Average Protection Against Expropriation Risk in 1985-1995")

table4_fs

## Instrumental Variable

In [73]:
with_continents = [["asia", "africa","other_cont" ],
                   ["lat_abst","asia", "africa","other_cont"]] 

iv_models = []

iv_columns = ["Base Sample (1)", "Base Sample (2)", "Base Sample without Neo-Europes (3)" ,
             "Base Sample without Neo-Europes (4)", "Base Sample without Neo-Africa (5)",
            "Base Sample without Neo-Africa (6)", "Base sample with continent dummies (7)", 
            "Base sample with continent dummies (8)", "Base sample, dependent ariable is log output per worker (8)"]


for sample in samples:
    reg1 = IV2SLS.from_formula(formula=f"logpgp95 ~ 1 + [avexpr ~ logem4]", data=sample).fit()
    iv_models.append(reg1)
    reg2 = IV2SLS.from_formula(formula=f"logpgp95 ~ 1 + [avexpr ~ logem4] + lat_abst", data=sample).fit()
    iv_models.append(reg2) 
    

for indep in with_continents:
    reg = IV2SLS.from_formula(formula=f"logpgp95 ~ 1 + [avexpr ~ logem4] + {'+'.join(indep)}", data=base_sample).fit()
    iv_models.append(reg)

reg_worker = IV2SLS.from_formula(formula="loghjypl ~ 1 + [avexpr ~ logem4]", data=base_sample.dropna(subset="loghjypl")).fit()
iv_models.append(reg_worker)


full = dict(zip(iv_columns, iv_models))
tab = compare(full, stars=True, precision="std_errors")
display(HTML(tab.summary.as_html()))

,Base Sample (1),Base Sample (2),Base Sample without Neo-Europes (3),Base Sample without Neo-Europes (4),Base Sample without Neo-Africa (5),Base Sample without Neo-Africa (6),Base sample with continent dummies (7),Base sample with continent dummies (8),"Base sample, dependent ariable is log output per worker (8)"
Dep. Variable,logpgp95,logpgp95,logpgp95,logpgp95,logpgp95,logpgp95,logpgp95,logpgp95,loghjypl
Estimator,IV-2SLS,IV-2SLS,IV-2SLS,IV-2SLS,IV-2SLS,IV-2SLS,IV-2SLS,IV-2SLS,IV-2SLS
No. Observations,64,64,60,60,37,37,64,64,61
Cov. Est.,robust,robust,robust,robust,robust,robust,robust,robust,robust
R-squared,0.1870,0.1025,-0.6877,-0.4918,0.5874,0.5886,0.2286,0.0108,-0.1518
Adj. R-squared,0.1739,0.0730,-0.7168,-0.5442,0.5756,0.5644,0.1763,-0.0745,-0.1714
F-statistic,28.754,28.333,10.163,13.536,49.934,50.806,36.376,28.443,24.958
P-value (F-stat),8.217e-08,7.041e-07,0.0014,0.0011,1.59e-12,9.282e-12,2.421e-07,2.981e-05,5.86e-07
==================,===========,===========,===========,===========,===========,===========,===========,===========,============
Intercept,1.9097,1.6918,-0.1412,0.1442,4.5539***,4.5623***,2.0324,1.4405,-8.3229***
